In [1]:
# ── CELL 1 — Install dependencies ────────────────────────────
!pip install roboflow ultralytics --quiet
print("✓ Done")

from roboflow import Roboflow
rf = Roboflow(api_key="yOnFYmDYTuLZyLo6iRP1")
project = rf.workspace("neudatasetoriginal").project("neu-steel-defect-dataset")
dataset = project.version(1).download("yolov8")

print(f"✓ Dataset downloaded to: {dataset.location}")
print("   Classes:", project.classes)

# After this cell runs, note the dataset.location path — use it in train.py

import os, shutil, random, glob
import xml.etree.ElementTree as ET

# Class names in NEU-DET
CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface',
           'rolled-in_scale', 'scratches']

def xml_to_yolo(xml_path, img_w, img_h, classes):
    """Convert Pascal VOC XML annotation to YOLO format lines."""
    tree  = ET.parse(xml_path)
    root  = tree.getroot()
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in classes:
            continue
        cls_id = classes.index(name)
        bb     = obj.find('bndbox')
        xmin   = float(bb.find('xmin').text)
        ymin   = float(bb.find('ymin').text)
        xmax   = float(bb.find('xmax').text)
        ymax   = float(bb.find('ymax').text)
        cx     = (xmin + xmax) / 2 / img_w
        cy     = (ymin + ymax) / 2 / img_h
        w      = (xmax - xmin) / img_w
        h      = (ymax - ymin) / img_h
        lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines

def build_yolo_dataset(neu_det_dir, out_dir, split=(0.7, 0.15, 0.15)):
    """
    Convert NEU-DET (XML) to YOLO folder structure.
    neu_det_dir: path to unzipped NEU-DET folder
    out_dir:     where to write the YOLO dataset
    """
    ann_dir = os.path.join(neu_det_dir, 'ANNOTATIONS')
    img_dir = os.path.join(neu_det_dir, 'IMAGES')

    # Get all xml files
    xml_files = sorted(glob.glob(os.path.join(ann_dir, '*.xml')))
    random.seed(42)
    random.shuffle(xml_files)

    n       = len(xml_files)
    n_train = int(n * split[0])
    n_val   = int(n * split[1])
    splits  = {
        'train': xml_files[:n_train],
        'val':   xml_files[n_train:n_train + n_val],
        'test':  xml_files[n_train + n_val:],
    }

    for split_name, files in splits.items():
        img_out = os.path.join(out_dir, 'images', split_name)
        lbl_out = os.path.join(out_dir, 'labels', split_name)
        os.makedirs(img_out, exist_ok=True)
        os.makedirs(lbl_out, exist_ok=True)

        for xml_path in files:
            stem   = os.path.splitext(os.path.basename(xml_path))[0]
            img_path = os.path.join(img_dir, stem + '.jpg')
            if not os.path.exists(img_path):
                img_path = os.path.join(img_dir, stem + '.bmp')

            # Get image size
            from PIL import Image
            img    = Image.open(img_path)
            w, h   = img.size

            # Convert annotation
            lines  = xml_to_yolo(xml_path, w, h, CLASSES)
            if not lines:
                continue

            # Copy image
            shutil.copy(img_path, os.path.join(img_out, os.path.basename(img_path)))

            # Write YOLO label file
            lbl_path = os.path.join(lbl_out, stem + '.txt')
            with open(lbl_path, 'w') as f:
                f.write('\n'.join(lines))

    # Write data.yaml
    yaml_content = f"""path: {out_dir}
train: images/train
val:   images/val
test:  images/test

nc: {len(CLASSES)}
names: {CLASSES}
"""
    with open(os.path.join(out_dir, 'data.yaml'), 'w') as f:
        f.write(yaml_content)

    print(f"✓ Dataset built at: {out_dir}")
    for s, files in splits.items():
        print(f"   {s}: {len(files)} images")

# Uncomment and run if using Option B (manual download):
# build_yolo_dataset(
#     neu_det_dir='/content/NEU-DET',
#     out_dir='/content/neu_det_yolo'
# )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 139.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpx2 2.4.0 requires idna>=3.18, but you have idna 3.7 which is incompatible.
✓ Done
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to neu-steel-defect-dataset-1 in yolov8:: 100%|██████████| 3610/3610 [00:00<00:00, 7474.65it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ Dataset downloaded to: /content/neu-steel-defect-dataset-1
   Classes: {'inclusion': 1011, 'pitted_surface': 432, 'rolled-in_scale': 628, 'patches': 878, 'scratches': 548, 'crazing': 689}


In [ ]:
# ================================================================
#  STEP 2 — Train YOLOv8 on NEU-DET
#  Run in Google Colab after prepare_dataset.py
#  Runtime: ~30–45 min on Colab T4 GPU
# ================================================================


# ── CELL 1 — Install ─────────────────────────────────────────
!pip install ultralytics --quiet
print("✓ Done")


# ── CELL 2 — Check GPU ───────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# Make sure you're on GPU: Runtime → Change runtime type → T4 GPU


# ── CELL 3 — Train ───────────────────────────────────────────
from ultralytics import YOLO

# Load YOLOv8 nano — small, fast, perfect for course project
# (switch to yolov8s.pt for better accuracy if you have time)
model = YOLO('yolov8n.pt')

results = model.train(
    data    = '/content/neu-steel-defect-dataset-1',  # ← path from prepare_dataset.py
                                                   # If using Roboflow: dataset.location + '/data.yaml'
    epochs  = 80,          # 50 epochs is enough; bump to 100 for higher mAP
    imgsz   = 640,         # standard YOLO input size
    batch   = 16,          # reduce to 8 if you get CUDA out-of-memory
    patience= 15,          # early stopping if no improvement for 15 epochs
    name    = 'steel_defect_v1',
    project = '/content/runs',
    device  = 0,           # GPU 0
    augment = True,        # built-in mosaic, flip, HSV augmentation
    degrees = 10,          # slight rotation augmentation
    scale   = 0.5,         # scale jitter
    fliplr  = 0.5,         # horizontal flip
    flipud  = 0.0,         # no vertical flip (defects have orientation)
    mosaic  = 1.0,         # mosaic augmentation (great for small dataset)
    verbose = True,
)

print("\n✓ Training complete!")
print(f"Best weights saved at: /content/runs/steel_defect_v1/weights/best.pt")

✓ Done
CUDA available: True
Device: Tesla T4
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/neu-steel-defect-dataset-1, degrees=10, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=steel_defect_v1, nbs=64, nms=False, opset=None, opt

In [ ]:
# ── CELL 4 — Evaluate on test set ────────────────────────────
best_model = YOLO('/content/runs/steel_defect_v1-4/weights/best.pt')

metrics = best_model.val(
    data   = '/content/neu-steel-defect-dataset-1/data.yaml',
    split  = 'test',
    imgsz  = 640,
    device = 0,
)

print("\n── TEST SET RESULTS ──────────────────────────────────")
print(f"  mAP@0.5:      {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"  Precision:    {metrics.box.p.mean():.4f}")
print(f"  Recall:       {metrics.box.r.mean():.4f}")

# Per-class breakdown
CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface',
           'rolled-in_scale', 'scratches']
print("\n── PER CLASS mAP@0.5 ────────────────────────────────")
for cls, ap in zip(CLASSES, metrics.box.ap50):
    print(f"  {cls:<22} {ap:.4f}")


# ── CELL 5 — Run inference on a sample image ─────────────────
import glob, random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Pick a random test image
test_imgs = glob.glob('/content/neu-steel-defect-dataset-1/test/images/*.jpg') + \
            glob.glob('/content/neu-steel-defect-dataset-1/test/images/*.bmp')
sample    = random.choice(test_imgs)

results   = best_model(sample, conf=0.25, imgsz=640)
result    = results[0]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original
axes[0].imshow(Image.open(sample), cmap='gray')
axes[0].set_title('Original image', fontsize=12)
axes[0].axis('off')

# With detections
plot_img = result.plot(labels=True, conf=True)
import cv2
plot_img  = cv2.cvtColor(plot_img, cv2.COLOR_BGR2RGB)
axes[1].imshow(plot_img)
axes[1].set_title('Detected defects', fontsize=12)
axes[1].axis('off')

plt.suptitle('Steel Surface Defect Detection — YOLOv8', fontsize=13)
plt.tight_layout()
plt.savefig('sample_detection.png', dpi=130, bbox_inches='tight')
plt.show()
print("✓ Sample saved → sample_detection.png")


# ── CELL 6 — Save weights for deployment ─────────────────────
import shutil, os

os.makedirs('/content/deployment', exist_ok=True)
shutil.copy('/content/runs/steel_defect_v1-4/weights/best.pt',
            '/content/deployment/best.pt')

print("✓ Weights saved → /content/deployment/best.pt")
print("\nNext steps:")
print("  1. Download best.pt from Colab Files panel")
print("  2. Put it in your app folder alongside app.py")
print("  3. Deploy to HuggingFace Spaces with requirements.txt")

In [ ]:
# ================================================================
#  Steel Surface Defect Detector — Streamlit Web App
#  Runs locally:  streamlit run app.py
#  Deploys to:    HuggingFace Spaces (see README.md)
# ================================================================

!pip install streamlit --quiet

import streamlit as st
from ultralytics import YOLO
from PIL import Image
import numpy as np
import cv2
import time
import os

# ── Page config ───────────────────────────────────────────────
st.set_page_config(
    page_title="Steel Surface Defect Detector",
    page_icon="🔬",
    layout="wide",
)

# ── Classes & colors (BGR for OpenCV) ─────────────────────────
CLASSES = {
    'crazing':         {'color': (255, 100,  50),  'desc': 'Network of fine surface cracks'},
    'inclusion':       {'color': ( 50, 200, 255),  'desc': 'Foreign material trapped in steel'},
    'patches':         {'color': ( 50, 255, 100),  'desc': 'Irregular blotchy surface areas'},
    'pitted_surface':  {'color': (255, 200,  50),  'desc': 'Small cavities / holes on surface'},
    'rolled-in_scale': {'color': (200,  50, 255),  'desc': 'Oxide scale rolled into surface'},
    'scratches':       {'color': ( 50, 100, 255),  'desc': 'Linear surface marks'},
}

# ── Load model (cached — only loads once) ─────────────────────
@st.cache_resource
def load_model():
    model_path = '/content/runs/steel_defect_v1-4/weights/best.pt' # Updated path
    if not os.path.exists(model_path):
        st.error("❌ best.pt not found. Train the model first using train.py in Google Colab, "
                 "then place best.pt in the same folder as this app.py file.")
        st.stop()
    return YOLO(model_path)

model = load_model()

# ── Header ────────────────────────────────────────────────────
st.title("🔬 Steel Surface Defect Detector")
st.markdown(
    "Upload an image of a steel surface and the model will detect and locate "
    "defects using a **YOLOv8** model trained on the NEU Surface Defect Database "
    "(1,800 images, 6 defect types)."
)

# ── Sidebar — controls ────────────────────────────────────────
with st.sidebar:
    st.header("Settings")
    conf_thresh = st.slider("Confidence threshold", 0.10, 0.90, 0.25, 0.05,
                            help="Only show detections above this confidence score")
    iou_thresh  = st.slider("IoU threshold (NMS)", 0.10, 0.90, 0.45, 0.05,
                            help="Controls overlap removal between boxes")
    show_labels = st.checkbox("Show class labels on image", value=True)
    show_conf   = st.checkbox("Show confidence scores on image", value=True)

    st.divider()
    st.subheader("Defect types")
    for cls, info in CLASSES.items():
        r, g, b = info['color'][2], info['color'][1], info['color'][0]
        st.markdown(
            f"<div style='display:flex;align-items:center;gap:8px;margin-bottom:6px;'>"
            f"<div style='width:14px;height:14px;border-radius:3px;"
            f"background:rgb({r},{g},{b});flex-shrink:0'></div>"
            f"<span style='font-size:13px;font-weight:500'>{cls}</span></div>"
            f"<p style='font-size:12px;color:gray;margin:-4px 0 8px 22px'>{info['desc']}</p>",
            unsafe_allow_html=True
        )

    st.divider()
    st.caption("Model: YOLOv8n — NEU-DET dataset\n6 classes · 1,800 training images")

# ── Upload ────────────────────────────────────────────────────
uploaded = st.file_uploader(
    "Upload a steel surface image",
    type=["jpg", "jpeg", "png", "bmp"],
    help="Works best with grayscale or near-grayscale steel surface images"
)

# ── Sample images (for demo without uploading) ────────────────
st.markdown("**Or try a sample:**")
sample_cols = st.columns(6)
sample_classes = list(CLASSES.keys())
selected_sample = None
for i, (col, cls) in enumerate(zip(sample_cols, sample_classes)):
    with col:
        if st.button(cls.replace('_', ' ').replace('-', ' '), key=f"sample_{i}"):
            selected_sample = cls
            st.info(f"Sample mode: showing {cls}. Upload a real image to detect on your own photos.")

# ── Run inference ─────────────────────────────────────────────
if uploaded is not None:
    img = Image.open(uploaded).convert('RGB')

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Original image")
        st.image(img, use_column_width=True)

    with col2:
        st.subheader("Detection result")

        with st.spinner("Running defect detection..."):
            t0      = time.time()
            results = model(
                np.array(img),
                conf   = conf_thresh,
                iou    = iou_thresh,
                imgsz  = 640,
                verbose= False,
            )
            elapsed = time.time() - t0

        result   = results[0]
        plot_img = result.plot(labels=show_labels, conf=show_conf, line_width=2)
        plot_img = cv2.cvtColor(plot_img, cv2.COLOR_BGR2RGB)

        st.image(plot_img, use_column_width=True)
        st.caption(f"Inference time: {elapsed*1000:.0f} ms")

    # ── Detection summary ──────────────────────────────────────
    boxes = result.boxes
    st.divider()

    if boxes is None or len(boxes) == 0:
        st.warning("⚠️ No defects detected above the confidence threshold. "
                   "Try lowering the confidence slider in the sidebar.")
    else:
        st.subheader(f"Detected {len(boxes)} defect(s)")

        # Count per class
        class_counts = {}
        detections   = []
        for box in boxes:
            cls_id   = int(box.cls[0])
            cls_name = model.names[cls_id]
            conf     = float(box.conf[0])
            xyxy     = box.xyxy[0].tolist()
            class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
            detections.append({
                'Defect type': cls_name,
                'Confidence':  f"{conf:.1%}",
                'Severity':    '🔴 High' if conf > 0.75 else '🟡 Medium' if conf > 0.50 else '🟢 Low',
                'Location':    f"({int(xyxy[0])}, {int(xyxy[1])}) → ({int(xyxy[2])}, {int(xyxy[3])})",
            })

        # Metric cards
        metric_cols = st.columns(len(class_counts))
        for col, (cls_name, count) in zip(metric_cols, class_counts.items()):
            with col:
                st.metric(label=cls_name.replace('_', ' ').title(), value=count)

        # Detail table
        import pandas as pd
        df = pd.DataFrame(detections)
        st.dataframe(df, use_container_width=True, hide_index=True)

        # Interpretation
        st.subheader("What this means")
        for cls_name in class_counts:
            if cls_name in CLASSES:
                st.info(f"**{cls_name.replace('_', ' ').title()}** — {CLASSES[cls_name]['desc']}")

else:
    # Landing state — show instructions
    st.info("⬆️ Upload an image above to start detecting defects.")
    st.markdown("""
    **How it works:**
    1. You upload a photo of a steel surface
    2. YOLOv8 scans the image for surface anomalies
    3. Bounding boxes are drawn around each defect with its type and confidence score
    4. The summary table lists every detection with severity and location

    **Detectable defects:** crazing · inclusion · patches · pitted surface · rolled-in scale · scratches
    """)

# ── Footer ─────────────────────────────────────────────────────
st.divider()
st.caption(
    "Model trained on NEU Surface Defect Database (Northeastern University, China) · "
    "YOLOv8n · IIT Kanpur AI/ML in Materials — Group Project"
)

In [ ]:
# Install required packages first
!pip install streamlit ultralytics opencv-python-headless --quiet

# Run the app with a public tunnel
!streamlit run app.py &>/content/logs.txt &

import time
time.sleep(5)  # wait for it to start

# Get public URL via localtunnel
!npm install -g localtunnel --quiet
!lt --port 8501